# Pre Processing and Tokenizing functions

In [ ]:
def load_calibration_texts(path):
    texts = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                texts.append(line)
    return texts

calib_texts = load_calibration_texts(
    "/content/drive/MyDrive/FYP/Datasets/wikitext/train_500.txt"
)

print(len(calib_texts))
print(calib_texts[0])


321
= Robert Boulter =


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import sys,os
sys.path.append(os.path.abspath(".."))

In [2]:
from Datasets.dataloader import get_loader

c:\Users\CT-PROJECT\Documents\Team10_FYP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataloader=get_loader("meta-llama/Llama-2-7b-hf",nsamples=68,seq_len=512)

<class 'str'>
False


In [ ]:
from transformers import AutoTokenizer
from huggingface_hub import notebook_login



tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Llama-2-7b-hf",
    use_fast=False
)


tokenizer.pad_token = tokenizer.eos_token


config.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [ ]:
import torch

def build_calib_dataloader(texts, tokenizer, seq_len=2048):
    tokenized = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=seq_len,
        return_tensors="pt"
    )
    return [(tokenized["input_ids"][i:i+1],)
            for i in range(tokenized["input_ids"].size(0))]


In [ ]:
calib_dataloader = build_calib_dataloader(
    calib_texts[:128],   # first 128 samples
    tokenizer,
    seq_len=256
)


In [ ]:
calib_dataloader

[(tensor([[   1,  353, 4755,  350, 5059,  357,  353,    2,    2,    2,    2,    2,
              2,    2,    2,    2,    2,    2,    2,    2,    2,    2,    2,    2,
              2,    2,    2,    2,    2,    2,    2,    2,    2,    2,    2,    2,
              2,    2,    2,    2,    2,    2,    2,    2,    2,    2,    2,    2,
              2,    2,    2,    2,    2,    2,    2,    2,    2,    2,    2,    2,
              2,    2,    2,    2,    2,    2,    2,    2,    2,    2,    2,    2,
              2,    2,    2,    2,    2,    2,    2,    2,    2,    2,    2,    2,
              2,    2,    2,    2,    2,    2,    2,    2,    2,    2,    2,    2,
              2,    2,    2,    2,    2,    2,    2,    2,    2,    2,    2,    2,
              2,    2,    2,    2,    2,    2,    2,    2,    2,    2,    2,    2,
              2,    2,    2,    2,    2,    2,    2,    2,    2,    2,    2,    2,
              2,    2,    2,    2,    2,    2,    2,    2,    2,    2,    2,    2,
    

Loading the LLama-7b model


In [4]:
import torch
from transformers import AutoModelForCausalLM

model_name = "meta-llama/Llama-2-7b-hf"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.eval()


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:15<00:00, 19.38it/s]
Some parameters are on the meta device because they were offloaded to the cpu.


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
   

In [ ]:
for name, param in model.named_parameters():
    if param.device.type != "cuda":
        print(name, param.device)
        break


Block layers in cpu and some offloaded

In [9]:
for i, block in enumerate(model.model.layers):
    print(i, next(block.parameters()).device)


0 cuda:0
1 cuda:0
2 cuda:0
3 cuda:0
4 cuda:0
5 cuda:0
6 cuda:0
7 cuda:0
8 cuda:0
9 cuda:0
10 cuda:0
11 cuda:0
12 cuda:0
13 cuda:0
14 cuda:0
15 cuda:0
16 cuda:0
17 cuda:0
18 cuda:0
19 cuda:0
20 cuda:0
21 cuda:0
22 cuda:0
23 cuda:0
24 meta
25 meta
26 meta
27 meta
28 meta
29 meta
30 meta
31 meta


attention weights

In [ ]:
block = model.model.layers[0]

print(block.self_attn.q_proj)
print(block.self_attn.k_proj)
print(block.self_attn.v_proj)
print(block.self_attn.o_proj)


Linear(in_features=4096, out_features=4096, bias=False)
Linear(in_features=4096, out_features=4096, bias=False)
Linear(in_features=4096, out_features=4096, bias=False)
Linear(in_features=4096, out_features=4096, bias=False)


act_scales → tells how big activations can get

act_shifts → tells where activations are centered

They are used inside to reduce quantization error



##Scaling and Shifting factors collect

In [5]:
import torch.nn as nn
import functools
from tqdm import tqdm


activation scales

In [6]:
def get_act_scales(model, dataloader, num_samples=128):
    model.eval()
    device = next(model.parameters()).device
    act_scales = {}

    def stat_tensor(name, tensor):
        hidden_dim = tensor.shape[-1]
        tensor = tensor.view(-1, hidden_dim).abs().detach()
        comming_max = torch.max(tensor, dim=0)[0].float().cpu()
        if name in act_scales:
            act_scales[name] = torch.max(act_scales[name], comming_max)
        else:
            act_scales[name] = comming_max

    def stat_input_hook(m, x, y, name):
        if isinstance(x, tuple):
            x = x[0]
        stat_tensor(name, x)

    hooks = []
    for name, m in model.named_modules():
        if isinstance(m, nn.Linear):
            hooks.append(
                m.register_forward_hook(
                    functools.partial(stat_input_hook, name=name))
            )

    for i in tqdm(range(num_samples)):
        model(dataloader[i][0].to(device))

    for h in hooks:
        h.remove()

    return act_scales


activation shifts

In [7]:
def get_act_shifts(model, dataloader, num_samples=128):
    model.eval()
    device = next(model.parameters()).device
    act_shifts = {}

    def stat_tensor(name, tensor):
        hidden_dim = tensor.shape[-1]
        tensor = tensor.view(-1, hidden_dim).detach()
        comming_max = torch.max(tensor, dim=0)[0].float().cpu()
        comming_min = torch.min(tensor, dim=0)[0].float().cpu()
        center = (comming_max + comming_min) / 2
        if name in act_shifts:
            act_shifts[name] = 0.99 * act_shifts[name] + 0.01 * center
        else:
            act_shifts[name] = center

    def stat_input_hook(m, x, y, name):
        if isinstance(x, tuple):
            x = x[0]
        stat_tensor(name, x)

    hooks = []
    for name, m in model.named_modules():
        if isinstance(m, nn.Linear):
            hooks.append(
                m.register_forward_hook(
                    functools.partial(stat_input_hook, name=name))
            )

    for i in tqdm(range(num_samples)):
        model(dataloader[i][0].to(device))

    for h in hooks:
        h.remove()

    return act_shifts


In [8]:
act_scales = get_act_scales(
    model,
    dataloader,
    num_samples=68
)

act_shifts = get_act_shifts(
    model,
    dataloader,
    num_samples=68
)


100%|██████████| 68/68 [07:03<00:00,  6.23s/it]


In [10]:
import os, torch

BASE = "../quant_stats_modified"
os.makedirs(f"{BASE}/act_scales", exist_ok=True)
os.makedirs(f"{BASE}/act_shifts", exist_ok=True)

torch.save(act_scales, f"{BASE}/act_scales/Llama-2-7b.pt")
torch.save(act_shifts, f"{BASE}/act_shifts/Llama-2-7b.pt")

print("Saved act_scales & act_shifts ✅")


Saved act_scales & act_shifts ✅


###Scale values of some layers

In [11]:
list(act_scales.keys())[:5]

['model.layers.0.self_attn.q_proj',
 'model.layers.0.self_attn.k_proj',
 'model.layers.0.self_attn.v_proj',
 'model.layers.0.self_attn.o_proj',
 'model.layers.0.mlp.gate_proj']

In [12]:
layer_name = 'model.layers.0.mlp.gate_proj'
scales = act_scales[layer_name]

print(type(scales))
print(scales.shape)


<class 'torch.Tensor'>
torch.Size([4096])


In [13]:
len(act_scales)


225

In [14]:
print(scales[:10])


tensor([0.1674, 0.1583, 0.1722, 0.1335, 0.1793, 0.1975, 0.1776, 0.1603, 0.2079,
        0.2025])


In [23]:
print("min :", scales.min().item())
print("max :", scales.max().item())
print("mean:", scales.mean().item())


min : 0.001499176025390625
max : 2.865234375
mean: 0.18694862723350525


In [22]:
shifts = act_shifts[layer_name]

print(shifts[:10])
print("mean shift:", shifts.mean().item())


tensor([ 0.0215, -0.0075,  0.0064, -0.0015, -0.0128, -0.0012, -0.0177,  0.0038,
         0.0197, -0.0130])
mean shift: 0.00023959433019626886


In [21]:
torch.isnan(scales).any(), torch.isinf(scales).any()


(tensor(False), tensor(False))

In [20]:
(scales <= 0).any()


tensor(False)

In [19]:
summary = []

for k, v in act_scales.items():
    summary.append((k, v.mean().item(), v.max().item()))

summary[:5]


[('model.layers.0.self_attn.q_proj', 0.06711015105247498, 4.97265625),
 ('model.layers.0.self_attn.k_proj', 0.06711015105247498, 4.97265625),
 ('model.layers.0.self_attn.v_proj', 0.06711015105247498, 4.97265625),
 ('model.layers.0.self_attn.o_proj', 0.04604993760585785, 0.7587890625),
 ('model.layers.0.mlp.gate_proj', 0.18694862723350525, 2.865234375)]